# 🧩 Feature Engineering

In this section, new features are engineered from the existing dataset to improve model performance. These features aim to capture temporal patterns, statistical trends, and contextual relationships influencing energy consumption.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("energydata_complete.csv")
df.head()

In [ ]:
# Convert date column to datetime
df['date'] = pd.to_datetime(df['date'])

df.head()

## ⏱️ Time-Based Features

Time-based features help capture daily and weekly consumption patterns. These include hour of the day, day of the week, weekend indicator, and peak usage hours.

In [ ]:
df['hour'] = df['date'].dt.hour
df['day_of_week'] = df['date'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].isin([5,6]).astype(int)

# Peak hours: Morning + Evening
df['is_peak_hour'] = df['hour'].isin([6,7,8,9,18,19,20,21]).astype(int)

df[['date','hour','day_of_week','is_weekend','is_peak_hour']].head()

## 📊 Rolling Statistics

Rolling statistics help capture short-term trends and variability in energy consumption.

In [ ]:
df['rolling_mean_6'] = df['Appliances'].rolling(6).mean()
df['rolling_std_6'] = df['Appliances'].rolling(6).std()

df['rolling_mean_12'] = df['Appliances'].rolling(12).mean()
df['rolling_std_12'] = df['Appliances'].rolling(12).std()

df[['Appliances','rolling_mean_6','rolling_std_6']].head(10)

## 🔁 Lag Features

Lag features introduce past values of energy consumption to capture temporal dependencies.

In [ ]:
df['lag_1'] = df['Appliances'].shift(1)
df['lag_2'] = df['Appliances'].shift(2)
df['lag_6'] = df['Appliances'].shift(6)
df['lag_12'] = df['Appliances'].shift(12)
df['lag_24'] = df['Appliances'].shift(24)
df['lag_48'] = df['Appliances'].shift(48)

df[['Appliances','lag_1','lag_6']].head(10)

## 🌡️ Interaction Feature

The temperature difference between indoor and outdoor environments influences heating and cooling energy usage.

In [ ]:
df['temp_diff'] = df['T2'] - df['T_out']

df[['T2','T_out','temp_diff']].head()

## 🔄 Fourier Features (Cyclical Encoding)

Time variables are cyclical in nature. Fourier transformations help encode these cyclical patterns effectively.

In [ ]:
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

df['day_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['day_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

df[['hour','hour_sin','hour_cos']].head()

In [ ]:
# Remove rows with NaN due to rolling and lag
df = df.dropna()

df.shape

---

# 📄 Feature Engineering — Justification Document

## Overall Philosophy

Energy consumption in residential buildings is inherently a **time-dependent, multi-variate process**. Raw sensor readings alone — such as instantaneous temperature or humidity — capture only a snapshot of the environment at a single point in time. They fail to represent the *dynamics* of how consumption evolves, the *periodicity* of human behaviour, or the *interactions* between environmental variables that truly drive appliance usage.

Our feature engineering strategy rests on **four pillars**:

1. **Temporal context** — encode *when* an observation occurs within daily and weekly cycles.
2. **Short-term memory** — give the model access to *recent history* through rolling statistics and lag values.
3. **Domain-informed interactions** — construct features that reflect known *physical relationships* (e.g., indoor vs. outdoor temperature).
4. **Cyclical encoding** — represent periodic variables without artificial discontinuities.

Together these pillars allow a downstream model to learn complex consumption patterns that would be invisible in the raw data.

---

## Feature-by-Feature Justification

### 1. Time-Based Features (`hour`, `day_of_week`, `is_weekend`, `is_peak_hour`)

| Feature | Reasoning |
|---------|-----------|
| `hour` | Energy usage follows strong **diurnal patterns** — cooking, lighting, and HVAC loads vary dramatically between morning, afternoon, and night. Extracting the hour lets the model learn these intra-day rhythms. |
| `day_of_week` | Consumption profiles differ between weekdays (occupants at work/school) and weekends/holidays (occupants at home). The day index captures this **weekly seasonality**. |
| `is_weekend` | A binary flag that simplifies the weekday-vs-weekend distinction. Many models benefit from an explicit **categorical split** rather than relying solely on the ordinal day index. |
| `is_peak_hour` | Utility grids and household routines exhibit well-documented **peak demand windows** (morning 6–9 h, evening 18–21 h). Flagging these hours gives the model a direct signal aligned with domain knowledge. |

### 2. Rolling Statistics (`rolling_mean_6/12`, `rolling_std_6/12`)

A single appliance reading is noisy. **Rolling means** over 6 and 12 time-steps (1 h and 2 h at the 10-minute sampling rate) smooth out measurement noise and reveal the **local trend** — is consumption rising, falling, or steady?

**Rolling standard deviations** over the same windows quantify **local volatility**. High volatility may indicate intermittent high-power appliances (ovens, washing machines) cycling on and off, which is predictive of near-future spikes.

Two window sizes (6 and 12) are chosen to capture both *very short-term* and *medium-term* dynamics without introducing excessive multicollinearity.

### 3. Lag Features (`lag_1` through `lag_48`)

Energy consumption is **auto-correlated**: the best predictor of current usage is often *recent past usage*. Lag features make this temporal dependency explicit.

| Lag | Time Offset | Rationale |
|-----|-------------|-----------|
| `lag_1` | 10 min | Captures the **immediate momentum** of consumption. |
| `lag_2` | 20 min | Adds a second short-term memory point to help distinguish transients from sustained loads. |
| `lag_6` | 1 h | Aligns with **hourly behavioural cycles** (e.g., a cooking session that started an hour ago). |
| `lag_12` | 2 h | Captures **medium-range** dependencies. |
| `lag_24` | 4 h | Spans roughly a quarter of the day — useful for catching **half-day patterns**. |
| `lag_48` | 8 h | Covers a full working/sleeping shift, linking current usage to the **same phase of the previous cycle**. |

Multiple lags at geometrically increasing intervals give the model a rich *temporal receptive field* without an explosion in feature count.

### 4. Interaction Feature (`temp_diff`)

The difference `T2 - T_out` (indoor dining-room temperature minus outdoor temperature) is a direct proxy for the **thermal load on the building envelope**. When the gap is large and positive, heating systems must work harder; when it is negative (hot weather), cooling loads rise. This single feature encodes a **physically meaningful interaction** that would otherwise require the model to learn implicitly from two separate columns.

### 5. Fourier / Cyclical Encoding (`hour_sin`, `hour_cos`, `day_sin`, `day_cos`)

Representing *hour = 0* and *hour = 23* as integers 0 and 23 misleads distance-based and gradient-based models into thinking these two points are far apart, when in reality they are **neighbours on a circle**.

Mapping each periodic variable to a **(sin, cos) pair** preserves the true circular geometry. The Euclidean distance between any two encoded points now faithfully reflects their **angular separation**, eliminating edge-of-period artefacts.

### 6. Dropping NaN Rows

Rolling and lag operations introduce **leading NaN values** (up to 48 rows for `lag_48`). Rather than impute these with potentially misleading values, we drop them outright — the loss of approximately 48 rows out of 19,735 (less than 0.25%) is negligible and keeps the dataset **clean and unbiased**.

---

## Summary

| Category | Features | Count |
|----------|----------|-------|
| Time-based | `hour`, `day_of_week`, `is_weekend`, `is_peak_hour` | 4 |
| Rolling statistics | `rolling_mean_6`, `rolling_std_6`, `rolling_mean_12`, `rolling_std_12` | 4 |
| Lag values | `lag_1`, `lag_2`, `lag_6`, `lag_12`, `lag_24`, `lag_48` | 6 |
| Interaction | `temp_diff` | 1 |
| Cyclical encoding | `hour_sin`, `hour_cos`, `day_sin`, `day_cos` | 4 |
| **Total new features** | | **19** |

These 19 engineered features complement the original 27 sensor columns to give the model a comprehensive view of temporal patterns, recent consumption history, environmental context, and cyclical rhythms — all grounded in domain knowledge of residential energy systems.
